# 09 · Improving the model with what the tools and the platform do

**Question:** riskmapjnr, as designed, had the best prediction test in notebook 08. Can the improved model take what the existing tools and the platform do differently, and do better on Verra's prediction test?

| Change | Taken from |
|---|---|
| Verra's 30 classes cut by equal forest area instead of the geometric series | riskmapjnr's best slicing |
| Local loss rates over 1.3, 2.7, 10 and 20 km, beside the 5 km one | riskmapjnr's moving window, at several sizes |
| Change in the 5 km loss rate (this window's three years against the three before) | – |
| Latitude and longitude, to learn regional levels | forestatrisk's spatial random effect for each 10 km cell |
| Palm-oil price at the window's start year (WFP market prices, inverse-distance weighted as the platform's `rasterize_market_points`) | The platform's adopted predictor, which it reads at a fixed 2018 |
| A regression forest on each cell's lost share of its forest, instead of lost against kept | Verra's allocation, which works in amounts |

Each change is one change from the improved model (16 predictors, trained on the shares; notebook 07). AUC and Verra's tests are the mean of three seeds, since one seed moves Verra's tests by 10–30 ha (notebook 05). The bar, as for every option so far: **a lower prediction error in all three windows, without a lower AUC**. The code is `scripts/improve.py`; results in `outputs/runs/improve_*`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "scripts"))
import model as MM
import improve as I
import tool_comparison as T

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
OUT = HERE.parent / "outputs" / "summary"
RERUN = False
runs = [MM.cached(I.split_name(s), lambda s=s: I.run_split(s), rerun=RERUN) for s in MM.SPLITS]
WINDOWS = [run["split"].split(" → ")[1] for run in runs]
designed = [MM.cached(T.split_name(s), lambda s=s: T.run_split(s)) for s in MM.SPLITS]
same = [MM.cached(T.split_name(s).replace("tools_", "tools_same_timing_"), lambda s=s: T.run_same_timing(s)) for s in MM.SPLITS]

## 1 · The borrowed changes

In [2]:
rows = []
for w, run in zip(WINDOWS, runs):
    for name, r in run["variants"].items():
        for slicing, label in (("geometric", ""), ("equal_area", ", equal-area classes")):
            if slicing == "equal_area" and name != "Improved model":
                continue
            rows.append({"Variant": name + label, "Test window": w, "AUC": np.mean(r["aucs"]),
                         "Fit MedAE (ha)": np.mean(r[slicing]["fit"]), "Prediction MedAE (ha)": np.mean(r[slicing]["prediction"])})
for w, d, f in zip(WINDOWS, designed, same):
    for label, src in (("riskmapjnr, same timing (notebook 08)", f), ("riskmapjnr as designed (notebook 08)", d)):
        r = src["tools"]["riskmapjnr (moving window)"]
        rows.append({"Variant": label, "Test window": w, "AUC": np.mean(r["aucs"]),
                     "Fit MedAE (ha)": r["fit_medae_ha"], "Prediction MedAE (ha)": r["prediction_medae_ha"]})
variants = pd.DataFrame(rows)
variants.round(4).to_csv(OUT / "09_variants.csv", index=False)
pred = variants.pivot(index="Variant", columns="Test window", values="Prediction MedAE (ha)")
pred = pred.loc[variants.Variant.unique()]
base = pred.loc["Improved model"]
summary = pd.DataFrame({**{f"Prediction {w}": pred[w] for w in WINDOWS},
                        "Prediction, mean": pred.mean(axis=1),
                        "Windows better than the improved model": (pred.lt(base, axis=1)).sum(axis=1),
                        "Fit, mean": variants.groupby("Variant", sort=False)["Fit MedAE (ha)"].mean(),
                        "AUC, mean": variants.groupby("Variant", sort=False)["AUC"].mean()})
print("Verra's tests (MedAE, ha) and AUC; mean of three seeds for the improved model and its variants")
summary.round(3)

Verra's tests (MedAE, ha) and AUC; mean of three seeds for the improved model and its variants


,Prediction 2017–2019,Prediction 2020–2022,Prediction 2022–2024,"Prediction, mean",Windows better than the improved model,"Fit, mean","AUC, mean"
Variant,,,,,,,
Improved model,226.852,171.150,127.655,175.219,0,123.038,0.752
"Improved model, equal-area classes",273.644,168.864,140.615,194.374,1,120.876,0.752
Loss rates at several distances,255.017,157.050,146.674,186.247,1,115.990,0.752
Loss trend,225.097,191.533,131.233,182.621,1,118.491,0.751
Location,220.725,189.800,134.632,181.719,1,115.406,0.752
Palm-oil price (platform),206.662,180.525,151.791,179.660,1,114.654,0.752
Trained for the amount of loss,348.456,181.941,176.766,235.721,0,174.566,0.735
"riskmapjnr, same timing (notebook 08)",280.054,149.922,150.052,193.343,1,164.258,0.750
riskmapjnr as designed (notebook 08),187.578,121.043,117.108,141.910,3,21.837,0.750


**Reading:**

* **None of the borrowed changes lowers the prediction error in every window.** Each is better in one window and worse in another, and training for the amount is worse in all three.
* **The palm-oil price (the platform's) and location lower the fit error** (115 against 123 ha) and leave AUC unchanged, but their prediction errors are mixed: better in the first window, worse in the other two.
* **Equal-area classes (riskmapjnr's slicing) do worse than the geometric classes VT0007 recommends:** 194 against 175 ha.
* **With the same timing, the improved model stays ahead of riskmapjnr** (175 against 193 ha). As designed, riskmapjnr leads in every window (142 ha).

## 2 · Why riskmapjnr leads as designed

With the same timing as the other models, riskmapjnr's prediction error rises from 142 to 193 ha (notebook 08, section 4): its lead comes from its calibration map, built from the calibration window's own loss. That map sets steep class rates, which put most of the next window's loss where loss just happened. These changes try to give the improved model the same effect, from its saved maps (first seed) without retraining. The last row builds the improved model's maps the way riskmapjnr builds its own.

In [3]:
adjust = [MM.cached(I.split_name(s, "improve_adjust"), lambda s=s: I.adjustments(s), rerun=RERUN) for s in MM.SPLITS]
fit_at_end = [MM.cached(I.split_name(s, "improve_fit_at_end"), lambda s=s: I.run_fit_at_end(s), rerun=RERUN) for s in MM.SPLITS]
rows = {}
for w, a, e in zip(WINDOWS, adjust, fit_at_end):
    for label, (auc, fit, prediction) in a.items():
        rows.setdefault(label, {})[w] = prediction
    rows.setdefault("Class rates from the improved model's end-of-calibration map (three seeds)", {})[w] = np.mean(e["geometric"]["prediction"])
adjusted = pd.DataFrame(rows).T
adjusted["Mean"] = adjusted.mean(axis=1)
adjusted["Windows better than the improved model"] = (adjusted[WINDOWS].lt(adjusted.loc["Improved model", WINDOWS], axis=1)).sum(axis=1)
adjusted.round(1).to_csv(OUT / "09_adjustments.csv")
print("Prediction MedAE (ha); the improved model's first seed")
adjusted.round(1)

Prediction MedAE (ha); the improved model's first seed


,2017–2019,2020–2022,2022–2024,Mean,Windows better than the improved model
Improved model,233.9,167.5,137.1,179.5,0
Risk smoothed over 1.3 km,220.4,169.6,135.3,175.1,2
Risk smoothed over 2.7 km,233.6,175.8,145.4,184.9,1
Risk smoothed over 5.1 km,267.2,161.9,166.2,198.5,1
Zero risk beyond riskmapjnr's distance threshold,236.3,164.3,141.4,180.7,1
Forest without loss within 5 km in a class of its own,229.0,166.0,136.7,177.2,3
"riskmapjnr's classes in pairs, each split in two by our risk",200.5,125.3,149.4,158.4,2
riskmapjnr's construction: our end-of-calibration risk as a 2.7 km surface for both maps,346.4,167.6,191.9,235.3,0
Class rates from the improved model's end-of-calibration map (three seeds),382.7,210.7,204.8,266.1,0


**Reading:**

* **Smoothing the risk, setting zero risk far from the edge, and a class of its own for quiet forest change little.** The quiet-forest class is lower in all three windows, but only by 0.4 to 5 ha, well within the 10–30 ha that the seed alone moves these tests, so it is not a gain.
* **Building the improved model's maps the way riskmapjnr builds its own makes them worse** (235 and 266 ha). At the end of the calibration window, the cells cleared during it carry their own loss in their predictors (their distance to recent loss is zero), so the top classes get loss rates near one. riskmapjnr's 5 km window dilutes a cell's own loss among 2,600 cells.
* **Only riskmapjnr's own classes help** (split by our risk: 158 ha). These are its classes as designed, built from the calibration window's own loss, so they bring that timing with them.

## 3 · What this means

* **No change taken from the tools or the platform improves the model on Verra's prediction test.** The improved model stays as it is: 16 predictors, trained on the shares, with geometric classes.
* **With the same timing for every model, it is the best single model:** prediction 175 ha, fit 123 ha, AUC 0.752. Its rank average with riskmapjnr is close (173 ha).
* **The only way found to reach riskmapjnr's prediction error is its calibration map**, built from the calibration window's own loss.
  * **If the same timing is kept for every model**, the improved model leads.
  * **If riskmapjnr's timing is accepted**, riskmapjnr as designed (142 ha) or its classes split by our risk (158 ha) lead. The improved model given that timing does worse (235–266 ha).
* **The platform's palm-oil price adds nothing to the prediction test** at its window-start year. Read at a fixed 2018, as on the platform, it would see the future in the earlier windows. If it is kept, read it at each window's start year in Verra's tests.
* **Next, notebook 10:** which timing to use, from the literature, from other scores, and with a version of the improved model for that timing. It recommends option A, with the rank average as the platform's risk map.